In [1]:
from five_safes_tes_workbench.workbench import Workbench

In [2]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parents[1] / "contingency-tables"))

from contingency_table_utils import ContingencyTable, aggregate_tables

## Full example for Contingency Table Analysis using the 5s-TES workbench

[The 5s-TES workbench](https://github.com/federated-research/5S-TES-Workbench) provides a set of tools for interacting with Five Safes TES.
You provide the workbench your credentials for connecting to Five Safes TES, and it will configure your connection to the submission layer, including writing TES messages to your specification and collecting results.
The full details of how to use the workbench can be found in its README.
Here we will not focus on those details, but on how to use it to carry out an analysis.

For this example, the configuration is held in a `config.yml` file, as described in [the workbench](https://github.com/federated-research/5S-TES-Workbench/blob/main/example-config.yml).

For this demonstration, if you have access to the University of Nottingham submission layer, the project configuration is:

- project: "DelphiDemo"
- tes_base_url: "https://api.5s-tes.federated-research.com"
- minio_sts_endpoint: "https://api.minio.5s-tes.federated-research.com/sts"
- minio_endpoint: "https://api.minio.5s-tes.federated-research.com"
- minio_output_bucket: "126104output"
- tres:
    - "Nottingham TRE 01"
    - "Nottingham TRE 02"

For authentication, you will need to either get an access token from the submission layer user interface, or ask your administrator for keycloak details.

In [3]:
wb = Workbench()

wb.validate(config_path="config.yml")

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Validation successful
INFO | Config: project='DelphiDemo' tes_base_url='https://api.5s-tes.federated-research.com/' minio_sts_endpoint='https://api.minio.5s-tes.federated-research.com/sts' minio_endpoint='https://api.minio.5s-tes.federated-research.com/' minio_output_bucket='126104output' tres=['Nottingham TRE 01', 'Nottingham TRE 02']
INFO | Auth mode: AuthMode.CREDENTIALS


## Define SQL query

The query checks the `person` table and the `condition_occurrence` table to classify each person by:

- whether they have primary malignant neoplasm of skin
- whether they have hypertension

The query then groups by these two categorical variables and returns the count for each combination. 

In [4]:
hypertension_neoplasm_query = """
WITH hypertension AS (
  SELECT
    person_id,
    CASE
      WHEN person_id IN (
        SELECT person_id
        FROM "DelphiDemo".condition_occurrence
        WHERE condition_concept_id = 320128
      ) THEN 'has_hypertension' ELSE 'no_hypertension' END AS hypertension_status
  FROM "DelphiDemo".person
)

SELECT
  CASE
    WHEN p.person_id IN (
      SELECT person_id
      FROM "DelphiDemo".condition_occurrence
      WHERE condition_concept_id = 139750
    ) THEN 'with'
    ELSE 'without'
    END AS neoplasm_status,
  hypertension.hypertension_status,
  COUNT(p.person_id) as n
FROM "DelphiDemo".person p
  JOIN hypertension ON p.person_id = hypertension.person_id
GROUP BY neoplasm_status, hypertension_status
"""


wb.build_tes.simple_sql(
    name="hypertension x neoplasm contingency table",
    query=hypertension_neoplasm_query
)

wb.submit()

INFO | Building TES task from template: 'simple_sql'
INFO | Resolving template: 'simple_sql'
INFO | TES Task built successfully
INFO | TES payload:
{
   "name": "hypertension x neoplasm contingency table",
   "description": "Simple SQL Task",
   "outputs": [
      {
         "url": "s3://",
         "path": "/outputs",
         "type": "DIRECTORY",
         "name": "Output",
         "description": "Output results"
      }
   ],
   "executors": [
      {
         "image": "harbor.federated-analytics.ac.uk/5s-tes-analysis-tools/5s-tes-analysis-tools-tre-sqlpg:1.0.0",
         "command": [
            "--Output=/outputs/output.csv",
            "--Query=\nWITH hypertension AS (\n  SELECT\n    person_id,\n    CASE\n      WHEN person_id IN (\n        SELECT person_id\n        FROM \"DelphiDemo\".condition_occurrence\n        WHERE condition_concept_id = 320128\n      ) THEN 'has_hypertension' ELSE 'no_hypertension' END AS hypertension_status\n  FROM \"DelphiDemo\".person\n)\n\nSELECT\n  CA

'1322'

## Fetch and prepare contingency table outputs

This step fetches the approved output files from the TES submission. 

The `fetch_outputs()` method returns the downloaded file paths grouped by TRE. The output paths are then collected into `contingency_paths`. Each CSV file is read with `pandas` and converted into a `ContingencyTable` object.

At this stage, each object represents the contingency table counts from one TRE. These tables can then be aggregated into a single combined contingency table.

In [6]:
paths = wb.fetch_outputs()
contingency_paths = [v[0] for k, v in paths.items()]

pre_contingency_tables = [ContingencyTable(pd.read_csv(path)) for path in contingency_paths]

INFO | Fetching token from keycloak...
INFO | Requesting keycloak token from https://drs-core-identity.azurewebsites.net/realms/Dare-Control/protocol/openid-connect/token
INFO | Keycloak token fetched successfully
INFO | Exchanging bearer token for MinIO credentials via STS (https://api.minio.5s-tes.federated-research.com/sts)
INFO | MinIO client initialised (endpoint=https://api.minio.5s-tes.federated-research.com/, secure=True)
INFO | Child task info: 1323, TRE: Nottingham TRE 01, status: Completed
INFO | Found 1 result object(s) for task 1323
INFO | Downloading result object: 1323/output.csv
INFO | Downloaded 1323/output.csv -> /home/mszag6/Code/5s-TES-notebooks/workbench-delphi/notebook-analysis/output/Nottingham TRE 01/1323/output.csv
INFO | Child task info: 1324, TRE: Nottingham TRE 02, status: Completed
INFO | Found 1 result object(s) for task 1324
INFO | Downloading result object: 1324/output.csv
INFO | Downloaded 1324/output.csv -> /home/mszag6/Code/5s-TES-notebooks/workbench-

In [20]:
dir(pre_contingency_tables)
pre_contingency_tables[1].data

,neoplasm_status,hypertension_status,n
0,without,no_hypertension,49012
1,with,no_hypertension,496
2,with,has_hypertension,7
3,without,has_hypertension,247


## Aggregate and display the final contingency table

This step combines the TRE-level contingency table outputs into one aggregated table.

The `aggregate_tables()` function adds together the counts from each TRE. The `["n"]` value is then selected because it contains the count data for each combination of neoplasm status and hypertension status.

The final output is the combined 2 by 2 contingency table used for interpretation and statistical testing.

In [26]:
aggregated = aggregate_tables(pre_contingency_tables)
contingency_table = aggregated.contingency_table["n"]

display(contingency_table)

hypertension_status,has_hypertension,no_hypertension
neoplasm_status,,
with,20,1019
without,517,97967


In [25]:
from scipy.stats import chi2_contingency, fisher_exact
#chisq = chi2_contingency(contingency_table)

ht_neoplasm_fisher = fisher_exact(contingency_table)

In [27]:
print (ht_neoplasm_fisher)

SignificanceResult(statistic=np.float64(3.719161843731196), pvalue=np.float64(1.4501688370524617e-06))
